# ISKOLARChat — RAGAS Evaluation on a free Colab GPU


## Before you run — TWO things:
1. **Enable the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**.
2. **Add your keys as Colab Secrets** (the 🔑 icon, left sidebar). Add these three and toggle **Notebook access** ON for each:
   - `QDRANT_URL`
   - `QDRANT_API_KEY`
   - `COHERE_API_KEY`



In [ ]:
# 1) Confirm the GPU is attached
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > T4 GPU, then re-run this cell"

In [ ]:
# 2) Upload your code zip: iskolarchat_backend_for_colab.zip
from google.colab import files
print("Click 'Choose Files' and pick iskolarchat_backend_for_colab.zip")
up = files.upload()
import zipfile, io, os
fname = [k for k in up if k.lower().endswith(".zip")][0]
with zipfile.ZipFile(io.BytesIO(up[fname])) as z:
    z.extractall("/content")
assert os.path.exists("/content/backend/eval/testset.csv"), "testset.csv missing - wrong zip?"
print("Extracted to /content/backend")

In [ ]:
# 3) Install Python dependencies (a few minutes)
!pip install -q -r /content/backend/requirements.txt
!pip install -q -r /content/backend/eval/requirements-eval.txt
print("Deps installed. (pip resolver warnings here are usually harmless.)")

> ⚠️ If the **smoke-test cell (6)** later fails with an *import error*, do
> **Runtime → Restart session**, then re-run from **cell 4 onward** — the model
> stays cached on disk, so the pull is instant.

In [ ]:
# 4) Install Ollama, start the server, pull the judge/generator model
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, os
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"  # big RAGAS prompts need this
subprocess.Popen(["ollama", "serve"])
time.sleep(6)
print("Pulling qwen2.5:14b-instruct (~9 GB, a few minutes)...")
!ollama pull qwen2.5:14b-instruct
print("Model ready.")
# If you hit out-of-memory or it is slow, change BOTH this line and cell 5
# to qwen2.5:7b-instruct.

In [ ]:
# 5) Point the eval at local Ollama + your Qdrant/Cohere secrets
import os
from google.colab import userdata
os.environ["QDRANT_URL"]        = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"]    = userdata.get("QDRANT_API_KEY")
os.environ["COHERE_API_KEY"]    = userdata.get("COHERE_API_KEY")
os.environ["QDRANT_COLLECTION"] = "iskolarchat_chunks"
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "ollama"
os.environ["LLM_MODEL"]    = "qwen2.5:14b-instruct"
os.environ["GRADER_MODEL"] = "qwen2.5:14b-instruct"
os.environ.setdefault("SUPABASE_URL", "http://localhost")
os.environ.setdefault("SUPABASE_SERVICE_ROLE_KEY", "noop")
print("Env set -> judge/generator =", os.environ["LLM_MODEL"])

In [ ]:
# 6) Smoke test: verify Ollama + Cohere + Qdrant all work
import sys
for p in ("/content/backend", "/content/backend/eval"):
    if p not in sys.path:
        sys.path.insert(0, p)
import _common  # installs the max_tokens cap + patches out the HITL DB write
from app.services import llm, embeddings, vectorstore
print("LLM  :", llm.chat([{"role": "user", "content": "Reply with exactly: OK"}])[0][:40])
print("Cohere embed dim:", len(embeddings.embed_query("test")))
print("Qdrant hits     :", len(vectorstore.semantic_search(embeddings.embed_query("admission requirements"), 3)))
print("SMOKE OK -> ready for the full run")

In [ ]:
# 7) Run the FULL evaluation (Objective 2 + Objective 4). ~30-60 min on a T4.
import os
os.chdir("/content/backend")
!python -u eval/ragas_eval.py --recollect --delay 0
!python -u eval/agent_eval.py
print("=== DONE ===")

In [ ]:
# 8) Package and download the results
import glob, zipfile, os
outs = sorted(set(
    glob.glob("/content/backend/eval/results_*.csv")
    + glob.glob("/content/backend/eval/summary.csv")
    + glob.glob("/content/backend/eval/collected_*.json")
))
with zipfile.ZipFile("/content/eval_results.zip", "w") as z:
    for f in outs:
        z.write(f, arcname=os.path.basename(f))
print("Packaged:", [os.path.basename(f) for f in outs])
from google.colab import files
files.download("/content/eval_results.zip")